# 06 — RQ3 Multiple Regression Analysis

This notebook continues directly from **Notebook 05 — RQ2 Correlation Analysis**.

Notebook 05 examined each facial and image characteristic separately. This
notebook estimates their **conditional associations** with the CR-FIQA score by
including all primary characteristics in one multiple linear regression model.

## Research question

> **RQ3: Which facial and image characteristics remain associated with the
> CR-FIQA score when all other included characteristics are considered
> simultaneously?**

## Main analysis

The primary model:

- excludes `age` from the main feature set because it is a protected
  characteristic;
- standardizes continuous and graded characteristics;
- retains binary characteristics in their original 0/1 coding;
- fits ordinary least squares regression;
- reports HC3 heteroskedasticity-robust standard errors;
- applies Benjamini–Hochberg false-discovery-rate correction;
- checks multicollinearity, residual behavior, heteroskedasticity, leverage, and
  influential observations;
- compares adjusted regression effects with the univariate results from
  Notebook 05.

Two sensitivity analyses are retained:

1. a model including `age`;
2. a model excluding observations above the conventional Cook's-distance
   threshold \(4/N\).

## Outputs

```text
results/06_rq3_regression_analysis/
├── figures/
└── tables/
```

## 1. Shared project setup

In [ ]:
from pathlib import Path

setup_candidates = [
    Path.cwd() / "00_colab_setup.ipynb",
    Path.cwd() / "notebooks" / "00_colab_setup.ipynb",
    Path.cwd().parent / "notebooks" / "00_colab_setup.ipynb",
    Path("/content/drive/MyDrive/FIQA_Project/notebooks/00_colab_setup.ipynb"),
    Path("/content/drive/MyDrive/FIQA_Project/00_colab_setup.ipynb"),
]

SETUP_NOTEBOOK = next(
    (path for path in setup_candidates if path.exists()),
    None,
)

if SETUP_NOTEBOOK is None:
    checked_paths = "\n".join(f"- {path}" for path in setup_candidates)
    raise FileNotFoundError(
        "00_colab_setup.ipynb could not be found.\n"
        "Keep all notebooks in the same notebooks/ directory or update "
        "setup_candidates.\n\n"
        f"Checked:\n{checked_paths}"
    )

print(f"Running setup notebook: {SETUP_NOTEBOOK}")
get_ipython().run_line_magic("run", f'"{SETUP_NOTEBOOK}"')

## 2. Imports and analysis configuration

In [ ]:
import json
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import statsmodels.api as sm

from scipy import stats
from sklearn.preprocessing import StandardScaler
from statsmodels.stats.diagnostic import het_breuschpagan
from statsmodels.stats.multitest import multipletests
from statsmodels.stats.outliers_influence import (
    variance_inflation_factor,
)

warnings.filterwarnings("ignore", category=UserWarning)

ALPHA = 0.05
FDR_METHOD = "fdr_bh"

TARGET = "cr_fiqa_score"
IDENTITY_COLUMN = "cls"
IMAGE_PATH_COLUMN = "index"
DEMOGRAPHIC_GROUP_COLUMN = "group"

INCLUDE_AGE_IN_PRIMARY_MODEL = False
RUN_AGE_SENSITIVITY_MODEL = True
RUN_INFLUENCE_SENSITIVITY_MODEL = True

## 3. Input and output paths

In [ ]:
DATA_FILE = PROJECT_PATH / "diveface_fiqa_merged.csv"

RQ2_RESULTS_FILE = (
    PROJECT_PATH
    / "results"
    / "05_rq2_correlation_analysis"
    / "tables"
    / "pearson_spearman_comparison.csv"
)

RESULTS_PATH = (
    PROJECT_PATH
    / "results"
    / "06_rq3_regression_analysis"
)
FIGURES_PATH = RESULTS_PATH / "figures"
TABLES_PATH = RESULTS_PATH / "tables"

for path in [RESULTS_PATH, FIGURES_PATH, TABLES_PATH]:
    path.mkdir(parents=True, exist_ok=True)

if not DATA_FILE.exists():
    raise FileNotFoundError(
        "Merged dataset not found. Run Notebook 01 first:\n"
        f"{DATA_FILE}"
    )

print(f"Dataset:        {DATA_FILE}")
print(f"RQ2 results:    {RQ2_RESULTS_FILE}")
print(f"Output folder:  {RESULTS_PATH}")

## 4. Load the merged dataset

The full exploratory analysis is contained in Notebook 02. Only the checks
required for regression are repeated here.

In [ ]:
df = pd.read_csv(DATA_FILE)

print(f"Rows:    {len(df):,}")
print(f"Columns: {df.shape[1]}")

## 5. Feature definitions

In [ ]:
continuous_features = [
    "smile",
    "moustache",
    "beard",
    "sideburns",
    "head_roll",
    "head_yaw",
    "head_pitch",
    "blur",
    "exposure",
    "noise",
]

binary_features = [
    "mask",
    "headWear",
    "glasses",
    "eye_makeup",
    "lip_makeup",
    "forehead_occluded",
    "eye_occluded",
    "mouth_occluded",
]

protected_numeric_features = ["age"]

primary_features = continuous_features + binary_features

if INCLUDE_AGE_IN_PRIMARY_MODEL:
    primary_features = primary_features + protected_numeric_features

required_columns = list(
    dict.fromkeys(
        primary_features
        + protected_numeric_features
        + [
            TARGET,
            IDENTITY_COLUMN,
            IMAGE_PATH_COLUMN,
            DEMOGRAPHIC_GROUP_COLUMN,
        ]
    )
)

missing_columns = [
    column
    for column in required_columns
    if column not in df.columns
]

if missing_columns:
    raise KeyError(
        f"Missing required columns: {missing_columns}"
    )

print(f"Primary predictors: {len(primary_features)}")
print(f"Age in primary model: {INCLUDE_AGE_IN_PRIMARY_MODEL}")

### Why demographic group is not included in the primary model

The primary RQ3 model focuses on facial and image characteristics. Demographic
group is retained as metadata for diagnostics and later group-specific analyses,
rather than used as a predictor here. `age` is examined separately in a
sensitivity model.

## 6. Data validation and complete-case sample

In [ ]:
working_df = df.copy()

numeric_columns = list(
    dict.fromkeys(
        primary_features
        + protected_numeric_features
        + [TARGET]
    )
)

for column in numeric_columns:
    working_df[column] = pd.to_numeric(
        working_df[column],
        errors="coerce",
    )

binary_validation_rows = []

for feature in binary_features:
    observed_values = sorted(
        working_df[feature].dropna().unique().tolist()
    )
    valid_binary = set(observed_values).issubset({0, 1})

    binary_validation_rows.append(
        {
            "Feature": feature,
            "Observed_Values": observed_values,
            "Valid_0_1_Coding": valid_binary,
            "Missing": int(working_df[feature].isna().sum()),
        }
    )

binary_validation = pd.DataFrame(binary_validation_rows)

invalid_binary_features = binary_validation.loc[
    ~binary_validation["Valid_0_1_Coding"],
    "Feature",
].tolist()

if invalid_binary_features:
    raise ValueError(
        "Unexpected values found in binary predictors: "
        f"{invalid_binary_features}"
    )

display(binary_validation)

In [ ]:
# The primary sample is defined only by variables used in the primary model.
# Missing age values therefore do not remove rows unless age is included.

primary_analysis_columns = primary_features + [TARGET]

rows_before = len(working_df)

analysis_df = (
    working_df
    .dropna(subset=primary_analysis_columns)
    .copy()
    .reset_index(drop=True)
)

rows_removed = rows_before - len(analysis_df)

predictor_variation = (
    analysis_df[primary_features]
    .nunique(dropna=True)
    .sort_values()
)

constant_predictors = predictor_variation.loc[
    predictor_variation <= 1
].index.tolist()

valid_primary_features = [
    feature
    for feature in primary_features
    if feature not in constant_predictors
]

print(f"Rows before cleaning: {rows_before:,}")
print(f"Rows removed:         {rows_removed:,}")
print(f"Rows used:            {len(analysis_df):,}")
print(f"Constant predictors:  {constant_predictors or 'None'}")

## 7. Descriptive statistics for the regression sample

In [ ]:
descriptive_statistics = (
    analysis_df[valid_primary_features + [TARGET]]
    .describe()
    .T
)

descriptive_statistics["missing"] = (
    analysis_df[valid_primary_features + [TARGET]]
    .isna()
    .sum()
)

descriptive_statistics["unique_values"] = (
    analysis_df[valid_primary_features + [TARGET]]
    .nunique()
)

display(descriptive_statistics)

## 8. Build the primary design matrix

Continuous and graded predictors are standardized to mean zero and standard
deviation one. Their coefficients therefore represent the expected CR-FIQA
change associated with a one-standard-deviation predictor increase, conditional
on all other predictors.

Binary predictors remain coded as 0 and 1. Their coefficients represent the
adjusted mean difference between state 1 and state 0.

In [ ]:
continuous_features_in_model = [
    feature
    for feature in continuous_features
    if feature in valid_primary_features
]

binary_features_in_model = [
    feature
    for feature in binary_features
    if feature in valid_primary_features
]

X_model = analysis_df[valid_primary_features].astype(float).copy()
y_model = analysis_df[TARGET].astype(float).copy()

continuous_scaler = StandardScaler()

if continuous_features_in_model:
    X_model[continuous_features_in_model] = (
        continuous_scaler.fit_transform(
            X_model[continuous_features_in_model]
        )
    )

X_model_with_constant = sm.add_constant(
    X_model,
    has_constant="add",
)

print(f"Design matrix: {X_model_with_constant.shape}")
print(f"Continuous/graded predictors: {continuous_features_in_model}")
print(f"Binary predictors: {binary_features_in_model}")

## 9. Fit OLS and HC3-robust inference

In [ ]:
ols_results = sm.OLS(
    y_model,
    X_model_with_constant,
).fit()

hc3_results = ols_results.get_robustcov_results(
    cov_type="HC3"
)

print(
    "Primary model fitted with conventional OLS coefficients "
    "and HC3-robust inference."
)

HC3 changes the estimated standard errors, confidence intervals, and p-values,
but not the OLS coefficient estimates themselves. HC3 is used as the primary
inference because residual variance may not be constant.

## 10. Tidy coefficient tables with FDR correction

In [ ]:
def build_coefficient_table(
    result_object,
    covariance_label,
):
    """Convert a statsmodels result object into a tidy table."""

    parameter_names = result_object.model.exog_names
    confidence_intervals = np.asarray(
        result_object.conf_int()
    )

    table = pd.DataFrame(
        {
            "Feature": parameter_names,
            "Coefficient": np.asarray(result_object.params),
            "Standard_Error": np.asarray(result_object.bse),
            "T_Value": np.asarray(result_object.tvalues),
            "P_Value": np.asarray(result_object.pvalues),
            "CI_Lower_95": confidence_intervals[:, 0],
            "CI_Upper_95": confidence_intervals[:, 1],
            "Covariance": covariance_label,
        }
    )

    non_intercept_mask = table["Feature"] != "const"
    adjusted_p_values = np.full(len(table), np.nan)
    significant_fdr = np.full(len(table), False, dtype=bool)

    valid_test_mask = (
        non_intercept_mask
        & table["P_Value"].notna()
    )

    if valid_test_mask.any():
        rejected, adjusted, _, _ = multipletests(
            table.loc[valid_test_mask, "P_Value"],
            alpha=ALPHA,
            method=FDR_METHOD,
        )

        adjusted_p_values[valid_test_mask] = adjusted
        significant_fdr[valid_test_mask] = rejected

    table["Adjusted_P_Value_FDR"] = adjusted_p_values
    table["Significant_Raw"] = table["P_Value"] < ALPHA
    table["Significant_FDR"] = significant_fdr
    table["Absolute_Coefficient"] = table["Coefficient"].abs()

    return (
        table
        .sort_values(
            "Absolute_Coefficient",
            ascending=False,
            na_position="last",
        )
        .reset_index(drop=True)
    )

In [ ]:
ols_coefficient_table = build_coefficient_table(
    ols_results,
    covariance_label="Conventional OLS",
)

hc3_coefficient_table = build_coefficient_table(
    hc3_results,
    covariance_label="HC3 robust",
)

display(
    hc3_coefficient_table.style.format(
        {
            "Coefficient": "{:.4f}",
            "Standard_Error": "{:.4f}",
            "T_Value": "{:.4f}",
            "P_Value": "{:.4g}",
            "Adjusted_P_Value_FDR": "{:.4g}",
            "CI_Lower_95": "{:.4f}",
            "CI_Upper_95": "{:.4f}",
            "Absolute_Coefficient": "{:.4f}",
        }
    )
)

## 11. Fully standardized coefficients

The primary model leaves binary variables in 0/1 units because this gives them a
direct adjusted mean-difference interpretation. A second model standardizes
**all** predictors so their coefficient magnitudes can be compared on one
common scale.

In [ ]:
X_fully_standardized = (
    analysis_df[valid_primary_features]
    .astype(float)
    .copy()
)

full_scaler = StandardScaler()

X_fully_standardized[valid_primary_features] = (
    full_scaler.fit_transform(
        X_fully_standardized[valid_primary_features]
    )
)

X_fully_standardized = sm.add_constant(
    X_fully_standardized,
    has_constant="add",
)

standardized_results = (
    sm.OLS(
        y_model,
        X_fully_standardized,
    )
    .fit()
    .get_robustcov_results(cov_type="HC3")
)

standardized_coefficient_table = build_coefficient_table(
    standardized_results,
    covariance_label="HC3 fully standardized",
)

display(
    standardized_coefficient_table.style.format(
        {
            "Coefficient": "{:.4f}",
            "Standard_Error": "{:.4f}",
            "P_Value": "{:.4g}",
            "Adjusted_P_Value_FDR": "{:.4g}",
            "CI_Lower_95": "{:.4f}",
            "CI_Upper_95": "{:.4f}",
        }
    )
)

## 12. Model-fit statistics

In [ ]:
fitted_values = ols_results.fittedvalues
residuals = ols_results.resid

model_fit_summary = pd.DataFrame(
    {
        "Metric": [
            "N",
            "R_squared",
            "Adjusted_R_squared",
            "F_statistic",
            "F_test_p_value",
            "AIC",
            "BIC",
            "In_sample_RMSE",
            "In_sample_MAE",
        ],
        "Value": [
            int(ols_results.nobs),
            ols_results.rsquared,
            ols_results.rsquared_adj,
            ols_results.fvalue,
            ols_results.f_pvalue,
            ols_results.aic,
            ols_results.bic,
            np.sqrt(np.mean(residuals ** 2)),
            np.mean(np.abs(residuals)),
        ],
    }
)

display(model_fit_summary)

The RMSE and MAE above are **in-sample descriptive fit measures**. They are not
comparable to the held-out predictive evaluation in Notebooks 03 and 04.

## 13. Multicollinearity analysis

In [ ]:
vif_rows = []

for feature_index, feature_name in enumerate(
    X_model_with_constant.columns
):
    vif_rows.append(
        {
            "Feature": feature_name,
            "VIF": variance_inflation_factor(
                X_model_with_constant.values,
                feature_index,
            ),
        }
    )

vif_results = (
    pd.DataFrame(vif_rows)
    .query("Feature != 'const'")
    .sort_values("VIF", ascending=False)
    .reset_index(drop=True)
)

vif_results["VIF_Interpretation"] = np.select(
    [
        vif_results["VIF"] >= 10,
        vif_results["VIF"] >= 5,
    ],
    [
        "high",
        "moderate",
    ],
    default="acceptable",
)

display(vif_results.style.format({"VIF": "{:.3f}"}))

## 14. Residual diagnostics

In [ ]:
influence = ols_results.get_influence()

standardized_residuals = (
    influence.resid_studentized_internal
)

diagnostic_summary = pd.DataFrame(
    {
        "Statistic": [
            "Residual mean",
            "Residual standard deviation",
            "Residual skewness",
            "Residual excess kurtosis",
        ],
        "Value": [
            residuals.mean(),
            residuals.std(ddof=1),
            stats.skew(residuals),
            stats.kurtosis(residuals),
        ],
    }
)

display(diagnostic_summary)

In [ ]:
plt.figure(figsize=(8, 5))
plt.scatter(fitted_values, residuals, alpha=0.35)
plt.axhline(0, linestyle="--")
plt.xlabel("Fitted CR-FIQA score")
plt.ylabel("Residual")
plt.title("Residuals vs. Fitted Values")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "residuals_vs_fitted.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
plt.figure(figsize=(8, 5))
plt.hist(residuals, bins=40)
plt.xlabel("Residual")
plt.ylabel("Frequency")
plt.title("Residual Distribution")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "residual_distribution.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
fig = sm.qqplot(
    residuals,
    line="45",
    fit=True,
)
plt.title("Normal Q–Q Plot of Residuals")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "residual_qq_plot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

Residual normality is not required for unbiased OLS coefficient estimates.
Because the dataset is large and heteroskedasticity is plausible, the HC3
results—not visual normality alone—are used for the main inference.

## 15. Breusch–Pagan heteroskedasticity test

In [ ]:
breusch_pagan_result = het_breuschpagan(
    residuals,
    X_model_with_constant,
)

heteroskedasticity_results = pd.DataFrame(
    {
        "Statistic": [
            "LM statistic",
            "LM p-value",
            "F statistic",
            "F p-value",
        ],
        "Value": breusch_pagan_result,
    }
)

display(heteroskedasticity_results)

A small Breusch–Pagan p-value indicates evidence of non-constant residual
variance and supports the decision to report HC3-robust inference.

## 16. Influence and Cook's distance

In [ ]:
cooks_distance = influence.cooks_distance[0]
leverage = influence.hat_matrix_diag

n_observations = len(analysis_df)
cooks_threshold = 4 / n_observations

influence_table = pd.DataFrame(
    {
        "dataset_row": analysis_df.index,
        "image_path": analysis_df[IMAGE_PATH_COLUMN],
        "identity": analysis_df[IDENTITY_COLUMN],
        "group": analysis_df[DEMOGRAPHIC_GROUP_COLUMN],
        "actual_cr_fiqa_score": y_model,
        "fitted_cr_fiqa_score": fitted_values,
        "residual": residuals,
        "standardized_residual": standardized_residuals,
        "leverage": leverage,
        "cooks_distance": cooks_distance,
    }
)

influence_table["Above_Cooks_4_over_N"] = (
    influence_table["cooks_distance"] > cooks_threshold
)

influence_table = (
    influence_table
    .sort_values("cooks_distance", ascending=False)
    .reset_index(drop=True)
)

print(f"Cook's-distance threshold: {cooks_threshold:.6f}")
print(
    "Observations above threshold:",
    int(influence_table["Above_Cooks_4_over_N"].sum()),
)

display(influence_table.head(20))

In [ ]:
plt.figure(figsize=(10, 5))
plt.scatter(
    np.arange(len(cooks_distance)),
    cooks_distance,
    alpha=0.45,
    s=16,
)
plt.axhline(
    cooks_threshold,
    linestyle="--",
    label="4/N threshold",
)
plt.xlabel("Regression-sample observation")
plt.ylabel("Cook's distance")
plt.title("Cook's Distance")
plt.legend()
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "cooks_distance.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 17. Coefficient visualizations

In [ ]:
plot_coefficient_data = (
    hc3_coefficient_table
    .query("Feature != 'const'")
    .sort_values("Coefficient")
)

lower_errors = (
    plot_coefficient_data["Coefficient"]
    - plot_coefficient_data["CI_Lower_95"]
)

upper_errors = (
    plot_coefficient_data["CI_Upper_95"]
    - plot_coefficient_data["Coefficient"]
)

plt.figure(figsize=(10, 8))
plt.errorbar(
    plot_coefficient_data["Coefficient"],
    np.arange(len(plot_coefficient_data)),
    xerr=np.vstack([lower_errors, upper_errors]),
    fmt="o",
    capsize=3,
)
plt.axvline(0, linestyle="--")
plt.yticks(
    np.arange(len(plot_coefficient_data)),
    plot_coefficient_data["Feature"],
)
plt.xlabel("Coefficient with 95% HC3 confidence interval")
plt.ylabel("Predictor")
plt.title("RQ3 Adjusted Regression Coefficients")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "hc3_coefficient_plot.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

In [ ]:
standardized_plot_data = (
    standardized_coefficient_table
    .query("Feature != 'const'")
    .sort_values("Coefficient")
)

plt.figure(figsize=(10, 8))
plt.barh(
    standardized_plot_data["Feature"],
    standardized_plot_data["Coefficient"],
)
plt.axvline(0, linewidth=1)
plt.xlabel("Fully standardized HC3 coefficient")
plt.ylabel("Predictor")
plt.title("Fully Standardized Regression Coefficients")
plt.tight_layout()
plt.savefig(
    FIGURES_PATH / "standardized_coefficients.png",
    dpi=300,
    bbox_inches="tight",
)
plt.show()

## 18. Compare Notebook 05 correlations with adjusted coefficients

This comparison distinguishes:

- an **unadjusted association** in Notebook 05;
- an **adjusted conditional association** in Notebook 06.

A coefficient may shrink, disappear, or change direction after adjustment
because predictors can share information.

In [ ]:
if not RQ2_RESULTS_FILE.exists():
    raise FileNotFoundError(
        "Notebook 05 comparison table was not found. "
        "Run Notebook 05 first:\n"
        f"{RQ2_RESULTS_FILE}"
    )

rq2_results = pd.read_csv(RQ2_RESULTS_FILE)

required_rq2_columns = {
    "Feature",
    "Pearson_R",
    "Pearson_FDR_P",
    "Pearson_Significant_FDR",
    "Spearman_Rho",
    "Spearman_FDR_P",
    "Spearman_Significant_FDR",
}

missing_rq2_columns = sorted(
    required_rq2_columns.difference(rq2_results.columns)
)

if missing_rq2_columns:
    raise KeyError(
        "Notebook 05 comparison table is missing columns: "
        f"{missing_rq2_columns}"
    )

rq3_comparison_source = (
    hc3_coefficient_table
    .query("Feature != 'const'")
    [
        [
            "Feature",
            "Coefficient",
            "P_Value",
            "Adjusted_P_Value_FDR",
            "Significant_FDR",
        ]
    ]
    .rename(
        columns={
            "Coefficient": "RQ3_HC3_Coefficient",
            "P_Value": "RQ3_P_Value",
            "Adjusted_P_Value_FDR": "RQ3_FDR_P",
            "Significant_FDR": "RQ3_Significant_FDR",
        }
    )
)

rq2_rq3_comparison = (
    rq2_results
    .merge(
        rq3_comparison_source,
        on="Feature",
        how="inner",
        validate="one_to_one",
    )
)

rq2_rq3_comparison["Pearson_and_RQ3_Same_Direction"] = (
    np.sign(rq2_rq3_comparison["Pearson_R"])
    == np.sign(rq2_rq3_comparison["RQ3_HC3_Coefficient"])
)

rq2_rq3_comparison["Significant_in_RQ2_and_RQ3"] = (
    rq2_rq3_comparison["Pearson_Significant_FDR"]
    & rq2_rq3_comparison["RQ3_Significant_FDR"]
)

rq2_rq3_comparison["Association_Status"] = np.select(
    [
        (
            rq2_rq3_comparison["Pearson_Significant_FDR"]
            & rq2_rq3_comparison["RQ3_Significant_FDR"]
        ),
        (
            rq2_rq3_comparison["Pearson_Significant_FDR"]
            & ~rq2_rq3_comparison["RQ3_Significant_FDR"]
        ),
        (
            ~rq2_rq3_comparison["Pearson_Significant_FDR"]
            & rq2_rq3_comparison["RQ3_Significant_FDR"]
        ),
    ],
    [
        "Significant before and after adjustment",
        "Unadjusted association only",
        "Adjusted association only",
    ],
    default="Not significant after FDR correction",
)

display(rq2_rq3_comparison)

## 19. Sensitivity model including age

In [ ]:
if RUN_AGE_SENSITIVITY_MODEL:
    age_model_features = list(
        dict.fromkeys(valid_primary_features + ["age"])
    )

    age_analysis_df = (
        working_df
        .dropna(subset=age_model_features + [TARGET])
        .copy()
        .reset_index(drop=True)
    )

    X_age_model = (
        age_analysis_df[age_model_features]
        .astype(float)
        .copy()
    )
    y_age_model = age_analysis_df[TARGET].astype(float)

    continuous_age_features = list(
        dict.fromkeys(
            [
                feature
                for feature in continuous_features
                if feature in age_model_features
            ]
            + ["age"]
        )
    )

    age_scaler = StandardScaler()
    X_age_model[continuous_age_features] = (
        age_scaler.fit_transform(
            X_age_model[continuous_age_features]
        )
    )

    X_age_model = sm.add_constant(
        X_age_model,
        has_constant="add",
    )

    age_sensitivity_results = (
        sm.OLS(y_age_model, X_age_model)
        .fit()
        .get_robustcov_results(cov_type="HC3")
    )

    age_sensitivity_table = build_coefficient_table(
        age_sensitivity_results,
        covariance_label="HC3 with age",
    )

    print(f"Age-sensitivity rows: {len(age_analysis_df):,}")
    display(age_sensitivity_table)

else:
    age_sensitivity_table = pd.DataFrame()
    print("Age sensitivity model disabled.")

The age model is a sensitivity analysis rather than the primary result. Its
purpose is to check whether adding age materially changes the direction or
statistical evidence of the facial and image-characteristic coefficients.

In [ ]:
if not age_sensitivity_table.empty:
    age_sensitivity_comparison = (
        hc3_coefficient_table
        .query("Feature != 'const'")
        [
            [
                "Feature",
                "Coefficient",
                "Adjusted_P_Value_FDR",
                "Significant_FDR",
            ]
        ]
        .rename(
            columns={
                "Coefficient": "Primary_Coefficient",
                "Adjusted_P_Value_FDR": "Primary_FDR_P",
                "Significant_FDR": "Primary_Significant_FDR",
            }
        )
        .merge(
            age_sensitivity_table
            .query("Feature not in ['const', 'age']")
            [
                [
                    "Feature",
                    "Coefficient",
                    "Adjusted_P_Value_FDR",
                    "Significant_FDR",
                ]
            ]
            .rename(
                columns={
                    "Coefficient": "Age_Adjusted_Coefficient",
                    "Adjusted_P_Value_FDR": "Age_Adjusted_FDR_P",
                    "Significant_FDR": "Age_Adjusted_Significant_FDR",
                }
            ),
            on="Feature",
            how="inner",
            validate="one_to_one",
        )
    )

    age_sensitivity_comparison["Same_Direction"] = (
        np.sign(age_sensitivity_comparison["Primary_Coefficient"])
        == np.sign(
            age_sensitivity_comparison["Age_Adjusted_Coefficient"]
        )
    )

    age_sensitivity_comparison["Coefficient_Change"] = (
        age_sensitivity_comparison["Age_Adjusted_Coefficient"]
        - age_sensitivity_comparison["Primary_Coefficient"]
    )

    display(age_sensitivity_comparison)

else:
    age_sensitivity_comparison = pd.DataFrame()

## 20. Sensitivity analysis excluding influential observations

In [ ]:
influential_sample_rows = influence_table.loc[
    influence_table["Above_Cooks_4_over_N"],
    "dataset_row",
].astype(int).tolist()

if RUN_INFLUENCE_SENSITIVITY_MODEL and influential_sample_rows:
    reduced_mask = ~analysis_df.index.isin(
        influential_sample_rows
    )

    X_reduced = X_model_with_constant.loc[reduced_mask]
    y_reduced = y_model.loc[reduced_mask]

    reduced_results = (
        sm.OLS(y_reduced, X_reduced)
        .fit()
        .get_robustcov_results(cov_type="HC3")
    )

    reduced_coefficient_table = build_coefficient_table(
        reduced_results,
        covariance_label=(
            "HC3 excluding observations above Cook's 4/N"
        ),
    )

    influence_sensitivity_comparison = (
        hc3_coefficient_table
        .query("Feature != 'const'")
        [
            [
                "Feature",
                "Coefficient",
                "Adjusted_P_Value_FDR",
                "Significant_FDR",
            ]
        ]
        .rename(
            columns={
                "Coefficient": "Full_Coefficient",
                "Adjusted_P_Value_FDR": "Full_FDR_P",
                "Significant_FDR": "Full_Significant_FDR",
            }
        )
        .merge(
            reduced_coefficient_table
            .query("Feature != 'const'")
            [
                [
                    "Feature",
                    "Coefficient",
                    "Adjusted_P_Value_FDR",
                    "Significant_FDR",
                ]
            ]
            .rename(
                columns={
                    "Coefficient": "Reduced_Coefficient",
                    "Adjusted_P_Value_FDR": "Reduced_FDR_P",
                    "Significant_FDR": "Reduced_Significant_FDR",
                }
            ),
            on="Feature",
            how="inner",
            validate="one_to_one",
        )
    )

    influence_sensitivity_comparison["Coefficient_Change"] = (
        influence_sensitivity_comparison["Reduced_Coefficient"]
        - influence_sensitivity_comparison["Full_Coefficient"]
    )

    influence_sensitivity_comparison["Same_Direction"] = (
        np.sign(
            influence_sensitivity_comparison["Full_Coefficient"]
        )
        == np.sign(
            influence_sensitivity_comparison["Reduced_Coefficient"]
        )
    )

    print(
        "Rows excluded:",
        len(influential_sample_rows),
    )
    display(influence_sensitivity_comparison)

else:
    reduced_coefficient_table = pd.DataFrame()
    influence_sensitivity_comparison = pd.DataFrame()

    if not RUN_INFLUENCE_SENSITIVITY_MODEL:
        print("Influence sensitivity model disabled.")
    else:
        print("No observations exceeded the Cook's-distance threshold.")

The \(4/N\) rule is a screening threshold, not an automatic deletion rule.
Therefore, the full model remains the primary analysis, while the reduced model
is used only to assess sensitivity.

## 21. Automated RQ3 summary

In [ ]:
rq3_summary = (
    hc3_coefficient_table
    .query("Feature != 'const'")
    [
        [
            "Feature",
            "Coefficient",
            "Standard_Error",
            "P_Value",
            "Adjusted_P_Value_FDR",
            "Significant_FDR",
            "CI_Lower_95",
            "CI_Upper_95",
            "Absolute_Coefficient",
        ]
    ]
    .copy()
)

rq3_summary["Direction"] = np.select(
    [
        rq3_summary["Coefficient"] > 0,
        rq3_summary["Coefficient"] < 0,
    ],
    ["positive", "negative"],
    default="none",
)

rq3_summary["Interpretation"] = np.where(
    rq3_summary["Significant_FDR"],
    (
        "Significant conditional association after HC3 inference "
        "and FDR correction."
    ),
    (
        "No significant conditional association after HC3 inference "
        "and FDR correction."
    ),
)

rq3_summary = (
    rq3_summary
    .sort_values("Absolute_Coefficient", ascending=False)
    .reset_index(drop=True)
)

display(rq3_summary)

In [ ]:
significant_rq3_features = rq3_summary.loc[
    rq3_summary["Significant_FDR"],
    "Feature",
].tolist()

print("Features significant after HC3 and FDR correction:")
print(significant_rq3_features)

print(f"\nAdjusted R²: {ols_results.rsquared_adj:.4f}")
print(f"Maximum VIF: {vif_results['VIF'].max():.3f}")
print(
    "Observations above Cook's threshold:",
    int(influence_table["Above_Cooks_4_over_N"].sum()),
)

## 22. Interpretation guidance

When reporting RQ3:

- use the **HC3 coefficient table** as the main inferential result;
- report FDR-adjusted p-values rather than relying only on raw p-values;
- distinguish continuous/graded coefficients from binary adjusted differences;
- use the fully standardized table only for magnitude comparison;
- compare RQ3 with Notebook 05 to identify associations that survive adjustment;
- treat influential-observation and age models as sensitivity checks;
- do not interpret coefficients as causal effects;
- do not compare the in-sample regression RMSE directly with the held-out ML
  RMSE from Notebooks 03 and 04.

Because multiple images may belong to the same identity, the observations are
not guaranteed to be fully independent. This limitation should be stated when
presenting inferential results.

## 23. Save tables and metadata

In [ ]:
binary_validation.to_csv(
    TABLES_PATH / "binary_feature_validation.csv",
    index=False,
)

descriptive_statistics.to_csv(
    TABLES_PATH / "descriptive_statistics.csv",
)

ols_coefficient_table.to_csv(
    TABLES_PATH / "ols_coefficients.csv",
    index=False,
)

hc3_coefficient_table.to_csv(
    TABLES_PATH / "hc3_robust_coefficients.csv",
    index=False,
)

standardized_coefficient_table.to_csv(
    TABLES_PATH / "standardized_hc3_coefficients.csv",
    index=False,
)

model_fit_summary.to_csv(
    TABLES_PATH / "model_fit_summary.csv",
    index=False,
)

vif_results.to_csv(
    TABLES_PATH / "vif_results.csv",
    index=False,
)

diagnostic_summary.to_csv(
    TABLES_PATH / "residual_diagnostic_summary.csv",
    index=False,
)

heteroskedasticity_results.to_csv(
    TABLES_PATH / "breusch_pagan_test.csv",
    index=False,
)

influence_table.to_csv(
    TABLES_PATH / "influential_observations.csv",
    index=False,
)

rq2_rq3_comparison.to_csv(
    TABLES_PATH / "rq2_rq3_comparison.csv",
    index=False,
)

rq3_summary.to_csv(
    TABLES_PATH / "rq3_summary.csv",
    index=False,
)

if not age_sensitivity_table.empty:
    age_sensitivity_table.to_csv(
        TABLES_PATH / "age_sensitivity_coefficients.csv",
        index=False,
    )

if not age_sensitivity_comparison.empty:
    age_sensitivity_comparison.to_csv(
        TABLES_PATH / "age_sensitivity_comparison.csv",
        index=False,
    )

if not influence_sensitivity_comparison.empty:
    influence_sensitivity_comparison.to_csv(
        TABLES_PATH / "influence_sensitivity_comparison.csv",
        index=False,
    )

print("All Notebook 06 tables were saved.")

In [ ]:
run_metadata = {
    "alpha": ALPHA,
    "fdr_method": FDR_METHOD,
    "target": TARGET,
    "primary_features": valid_primary_features,
    "continuous_features_in_model": continuous_features_in_model,
    "binary_features_in_model": binary_features_in_model,
    "age_in_primary_model": INCLUDE_AGE_IN_PRIMARY_MODEL,
    "age_sensitivity_run": RUN_AGE_SENSITIVITY_MODEL,
    "influence_sensitivity_run": RUN_INFLUENCE_SENSITIVITY_MODEL,
    "primary_sample_rows": int(len(analysis_df)),
    "adjusted_r_squared": float(ols_results.rsquared_adj),
    "maximum_vif": float(vif_results["VIF"].max()),
    "cooks_threshold": float(cooks_threshold),
    "observations_above_cooks_threshold": int(
        influence_table["Above_Cooks_4_over_N"].sum()
    ),
    "significant_features_fdr": significant_rq3_features,
}

with (
    RESULTS_PATH / "run_metadata.json"
).open("w", encoding="utf-8") as file:
    json.dump(run_metadata, file, indent=2)

print("Saved run metadata.")

## 24. Saved-file summary

In [ ]:
print("Notebook 06 result files:")

for file_path in sorted(RESULTS_PATH.rglob("*")):
    if file_path.is_file():
        print("-", file_path.relative_to(RESULTS_PATH))

## 25. Transition to later notebooks

Notebook 05 provided unadjusted associations, while Notebook 06 provides
adjusted conditional associations and regression diagnostics.

The exported HC3 coefficient table, standardized coefficients, RQ2–RQ3
comparison, and sensitivity tables can be used in later notebooks to compare:

- statistical association;
- predictive feature importance;
- demographic consistency;
- model-explanation rankings.

These analyses answer different questions and should be interpreted together
rather than treated as interchangeable.